# Model Cards: Documentación Responsable de Modelos IA

> Aprende a crear Model Cards completas siguiendo el estándar de Mitchell et al. (2019)
> y automatiza su generación con Claude para cualquier modelo.

## Objetivos
- Comprender la estructura de una Model Card estándar
- Generar automáticamente Model Cards con Claude
- Documentar también el dataset con Datasheets for Datasets
- Exportar la documentación como archivo markdown

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic scikit-learn pandas --quiet

## 2. Estructura estándar de una Model Card

In [ ]:
import anthropic
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import pandas as pd
import json

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("""=== ESTRUCTURA DE UNA MODEL CARD (Mitchell et al. 2019) ===

1. Detalles del modelo
   - Nombre, versión, tipo de arquitectura, fecha
   - Organización responsable
   - Licencia

2. Uso previsto
   - Casos de uso primarios
   - Usuarios objetivo
   - Casos de uso FUERA del alcance

3. Factores
   - Variables relevantes (demografía, condiciones)
   - Factores de evaluación

4. Métricas
   - Métricas de rendimiento
   - Umbrales de decisión

5. Datos de evaluación
   - Datasets usados para evaluación
   - Motivación y preprocesamiento

6. Datos de entrenamiento
   - Descripción del dataset de entrenamiento

7. Análisis cuantitativo
   - Resultados desagregados por subgrupos

8. Consideraciones éticas
   - Riesgos identificados
   - Medidas de mitigación

9. Advertencias y recomendaciones
   - Limitaciones conocidas
   - Recomendaciones de uso
""")

## 3. Entrenar modelo de ejemplo y recopilar métricas

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Modelo de clasificación de spam de ejemplo
X, y = make_classification(n_samples=1000, n_features=10, n_classes=2,
                            n_informative=6, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

report = classification_report(y_test, y_pred, output_dict=True)
cm = confusion_matrix(y_test, y_pred)

METRICAS_MODELO = {
    "accuracy": round(report["accuracy"], 4),
    "precision_clase_1": round(report["1"]["precision"], 4),
    "recall_clase_1": round(report["1"]["recall"], 4),
    "f1_clase_1": round(report["1"]["f1-score"], 4),
    "falsos_positivos": int(cm[0][1]),
    "falsos_negativos": int(cm[1][0])
}

print("=== MÉTRICAS DEL MODELO ===")
for k, v in METRICAS_MODELO.items():
    print(f"  {k}: {v}")

## 4. Generar Model Card con Claude

In [ ]:
def generar_model_card(nombre_modelo: str, tarea: str, descripcion_datos: str,
                       metricas: dict, limitaciones: list[str], uso_previsto: str) -> str:
    """Genera una Model Card completa en markdown usando Claude."""
    prompt = f"""Genera una Model Card profesional y completa en formato Markdown.
Sigue el estándar de Mitchell et al. (2019). Incluye todas las secciones estándar.

Información del modelo:
- Nombre: {nombre_modelo}
- Tarea: {tarea}
- Descripción de los datos: {descripcion_datos}
- Métricas: {json.dumps(metricas, indent=2)}
- Limitaciones conocidas: {', '.join(limitaciones)}
- Uso previsto: {uso_previsto}

La Model Card debe ser:
- Clara y no técnica donde sea posible
- Honesta sobre limitaciones
- Con sección explícita de casos de uso NO recomendados
- En español"""

    return client.messages.create(
        model=MODELO, max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

model_card = generar_model_card(
    nombre_modelo="ClasificadorSpam-v1.0",
    tarea="Clasificación binaria de emails spam/no-spam",
    descripcion_datos="Dataset de 1000 emails etiquetados manualmente, 50% spam, 50% legítimos",
    metricas=METRICAS_MODELO,
    limitaciones=["No funciona bien con spam en otros idiomas",
                  "Puede generar falsos positivos en emails de marketing legítimo",
                  "Entrenado solo con emails en castellano"],
    uso_previsto="Filtro de spam para sistemas de email corporativo en español"
)

print(model_card[:800] + "\n...")
print(f"\n[Model Card completa: {len(model_card)} caracteres]")

## 5. Exportar y Datasheet for Datasets

In [ ]:
# Exportar Model Card como archivo markdown
with open("model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card)
print("✓ Model Card exportada a 'model_card.md'")

# Generar también un Datasheet for Datasets resumido
def generar_datasheet(nombre_dataset: str, descripcion: str, origen: str,
                      sesgos_potenciales: list[str]) -> str:
    """Genera un Datasheet for Datasets simplificado."""
    prompt = f"""Genera un Datasheet for Datasets (Gebru et al. 2018) resumido en Markdown.
Incluye: motivación, composición, proceso de recopilación, preprocesamiento, usos y limitaciones.

Dataset: {nombre_dataset}
Descripción: {descripcion}
Origen: {origen}
Sesgos potenciales: {', '.join(sesgos_potenciales)}

Formato markdown, conciso (máx 500 palabras), en español."""

    return client.messages.create(
        model=MODELO, max_tokens=800,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

datasheet = generar_datasheet(
    nombre_dataset="EmailSpam-ES-1K",
    descripcion="1000 emails en castellano etiquetados como spam o legítimos",
    origen="Recopilados de buzones corporativos con consentimiento explícito",
    sesgos_potenciales=["Sobre-representación de sectores tecnológicos",
                        "Solo emails en castellano peninsular",
                        "Período de recopilación: 2023-2024"]
)

with open("datasheet.md", "w", encoding="utf-8") as f:
    f.write(datasheet)
print("✓ Datasheet exportado a 'datasheet.md'")
print("\nVista previa del Datasheet:")
print(datasheet[:500] + "...")